# Airbus Ship Detection - Complete Colab Experiment

This notebook runs the final experiment using the reproducible 6,000-image Airbus subset stored in Google Drive.

It can run the traditional ML benchmark, train Simple CNN and DenseNet-169, evaluate all models, and save results back to Drive.

**Drive folder supplied:** `1SrNmbhCX2K-hezpJ5pH9MoDQsgBCBqkO`

Keep the subset split fixed: 4,000 train, 1,000 validation, 1,000 test.


In [1]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/kmanu28/Machine-Learning-MiniProject.git'
REPO_DIR = Path('/content/Machine-Learning-MiniProject')
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_PROJECT = DRIVE_ROOT / 'Machine-Learning-MiniProject'
DRIVE_FOLDER_ID = '1SrNmbhCX2K-hezpJ5pH9MoDQsgBCBqkO'

if REPO_DIR.exists():
    !git -C /content/Machine-Learning-MiniProject pull --ff-only
else:
    !git clone https://github.com/kmanu28/Machine-Learning-MiniProject.git /content/Machine-Learning-MiniProject

%cd /content/Machine-Learning-MiniProject
!pip install -q scipy scikit-learn pandas matplotlib seaborn pillow opencv-python-headless scikit-image joblib


Mounted at /content/drive
Cloning into '/content/Machine-Learning-MiniProject'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 203 (delta 18), reused 196 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 9.40 MiB | 25.96 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/Machine-Learning-MiniProject


In [2]:
# Confirm GPU and locate the prepared subset in Drive
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

candidates = [
    DRIVE_PROJECT / 'data' / 'subset_6000',
    DRIVE_ROOT / 'subset_6000',
    DRIVE_ROOT / 'data' / 'subset_6000',
]
subset_candidates = [p for p in candidates if (p / 'manifest.csv').exists()]
if not subset_candidates:
    # Search one level below MyDrive without scanning the full Drive recursively.
    subset_candidates = [p.parent for p in DRIVE_ROOT.glob('**/manifest.csv') if p.parent.name == 'subset_6000']
if not subset_candidates:
    raise FileNotFoundError('Could not find subset_6000/manifest.csv in Google Drive. Put the local data/subset_6000 folder in Drive and rerun this cell.')

DATA_DIR = subset_candidates[0]
print('Using dataset:', DATA_DIR)
for split in ('train', 'val', 'test'):
    print(split, len(list((DATA_DIR / split).rglob('*.jpg'))) + len(list((DATA_DIR / split).rglob('*.png'))))


CUDA available: True
Device: Tesla T4
Using dataset: /content/drive/MyDrive/Machine-Learning-MiniProject/data/subset_6000
train 4000
val 1000
test 1000


## Optional: rerun traditional ML

The traditional experiment is already completed locally and its results are in the repository. Leave `RUN_TRADITIONAL = False` unless you want to reproduce it in Colab.


In [ ]:
RUN_TRADITIONAL = False
if RUN_TRADITIONAL:
    !python scripts/run_subset_experiment.py --data-dir "$DATA_DIR"
else:
    print('Skipping traditional ML; using the completed repository results.')


## Train both CNNs

The training script uses the fixed train/validation/test split, saves the best checkpoints, and reports held-out test metrics.


In [ ]:
# Start with 15 epochs. Use 8 as batch size if Colab GPU memory is limited.
EPOCHS = 15
BATCH_SIZE = 16
!python src/train_cnn.py --data-dir "{DATA_DIR}" --epochs {EPOCHS} --batch-size {BATCH_SIZE}


Loading 640 ship images and 3360 no-ship images...


In [ ]:
# Inspect CNN results
from IPython.display import display, Image

with open('models/cnn_results.json') as f:
    cnn_results = json.load(f)
print(json.dumps(cnn_results, indent=2))
display(Image(filename='models/cnn_training_curves.png'))


In [ ]:
# Persist CNN artifacts in Drive
RESULTS_DIR = DRIVE_PROJECT / 'models'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in ['sim_cnn.pth', 'tl_cnn.pth', 'cnn_results.json', 'cnn_training_curves.png']:
    source = Path('models') / name
    if source.exists():
        shutil.copy2(source, RESULTS_DIR / name)
        print('Saved', RESULTS_DIR / name)


## Final comparison table

This cell combines the traditional ML results with the CNN results for the report.


In [ ]:
# Print compact final comparison
traditional = json.load(open('models/experiment_results.json'))
print('Best traditional held-out result:')
best = max(((v['accuracy'], feature, model) for feature, row in traditional['table_1_comparison'].items() for model, v in row.items()), key=lambda x: x[0])
print({'model': best[2], 'features': best[1], 'accuracy': best[0]})
print('
CNN test results:')
for name, result in cnn_results.items():
    print(name, result.get('test_metrics', {}))


## Download artifacts

Download these files from `models/` and copy them into the local project:

- `cnn_results.json`
- `cnn_training_curves.png`
- `sim_cnn.pth`
- `tl_cnn.pth`

Then update the final report tables with the actual CNN metrics.
